# 08: Data Utilities — Sample Generation, File Hashing & Downloads

This notebook demonstrates three data utility modules in siege_utilities:

1. **Sample Datasets** — Load bundled sample data
2. **Synthetic Population** — Generate fake demographic data
3. **Synthetic Businesses** — Generate fake business records
4. **Synthetic Housing** — Generate fake housing data
5. **Spatial Data Integration** — Combine synthetic data with geographic boundaries
6. **File Integrity & Hashing** — Compute and verify file hashes for data integrity
7. **Remote File Downloads** — Download files with retry logic and metadata inspection

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install faker requests
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import siege_utilities as su
from siege_utilities.data.sample_data import (
    generate_synthetic_population,
    generate_synthetic_businesses,
    generate_synthetic_housing,
    load_sample_data,
    list_available_datasets
)
from IPython.display import display

import pandas as pd
import numpy as np

su.configure_shared_logging(level="INFO")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_08"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print("Data utilities notebook ready.")

## 1. Available Sample Datasets

List and load bundled sample datasets.

In [ ]:
# List available datasets
datasets = list_available_datasets()
print(f"Found {len(datasets)} available sample datasets:")
for name, info in datasets.items():
    desc = info.get('description', 'No description')
    print(f"  - {name}: {desc}")

In [ ]:
# Load a sample dataset (if available)
if datasets:
    first_name = next(iter(datasets))
    sample = load_sample_data(first_name)
    print(f"Loaded '{first_name}': {sample.shape[0]} rows, {sample.shape[1]} columns")
    display(sample.head())
else:
    print("No bundled datasets available.")

## 2. Generate Synthetic Population

Create fake demographic data for testing and development.

In [ ]:
# Generate synthetic population
population = generate_synthetic_population(size=100)
print(f"Generated {len(population)} synthetic population records")
print(f"Columns: {list(population.columns)}")
display(population.head())

In [ ]:
# Analyze the synthetic population
if 'age' in population.columns:
    print(f"Age: mean={population['age'].mean():.1f}, min={population['age'].min()}, max={population['age'].max()}")

if 'gender' in population.columns:
    print("Gender distribution:")
    display(population['gender'].value_counts().to_frame())

In [ ]:
# Generate larger population for more realistic analysis
large_pop = generate_synthetic_population(size=1000)
memory_kb = large_pop.memory_usage(deep=True).sum() / 1024
print(f"Generated {len(large_pop)} people ({memory_kb:.1f} KB)")

## 3. Generate Synthetic Businesses

Create fake business records for testing and development.

In [ ]:
# Generate synthetic businesses
businesses = generate_synthetic_businesses(business_count=50)
print(f"Generated {len(businesses)} synthetic businesses")
print(f"Columns: {list(businesses.columns)}")
display(businesses.head())

In [ ]:
# Analyze business types (if column exists)
type_cols = [c for c in businesses.columns if 'type' in c.lower() or 'industry' in c.lower()]
if type_cols:
    col = type_cols[0]
    print(f"Business distribution by {col}:")
    display(businesses[col].value_counts().head(10).to_frame())

## 4. Generate Synthetic Housing

Create fake housing/property records.

In [ ]:
# Generate synthetic housing data
housing = generate_synthetic_housing(housing_count=100)
print(f"Generated {len(housing)} synthetic housing records")
print(f"Columns: {list(housing.columns)}")
display(housing.head())

In [ ]:
# Price distribution (if available)
price_cols = [c for c in housing.columns if 'price' in c.lower() or 'value' in c.lower()]
if price_cols:
    col = price_cols[0]
    print(f"Price distribution ({col}):")
    print(f"  Mean:   ${housing[col].mean():,.0f}")
    print(f"  Median: ${housing[col].median():,.0f}")
    print(f"  Min:    ${housing[col].min():,.0f}")
    print(f"  Max:    ${housing[col].max():,.0f}")

## 5. Combining Synthetic Data with Spatial Data

Use synthetic data with geographic boundaries for testing.

In [ ]:
from siege_utilities.geo.spatial_data import get_census_boundaries
import matplotlib.pyplot as plt

# Get CA counties
counties = get_census_boundaries(year=2020, geographic_level='county', state_fips='06')
ca_counties = counties[counties['statefp'] == '06'].copy()

# Assign random synthetic population counts to counties
np.random.seed(42)
ca_counties['synthetic_population'] = np.random.randint(10000, 1000000, size=len(ca_counties))

print(f"Assigned synthetic population to {len(ca_counties)} counties")
print(f"Total synthetic population: {ca_counties['synthetic_population'].sum():,}")

In [ ]:
# Visualize
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 12))
ca_counties.plot(
    column='synthetic_population',
    ax=ax,
    legend=True,
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.3
)
ax.set_title('Synthetic Population by County')
ax.axis('off')
plt.tight_layout()
pop_map_path = OUTPUT_DIR / 'synthetic_population_map.png'
plt.savefig(str(pop_map_path), dpi=100, bbox_inches='tight')
plt.show()
plt.close()

print(f"Map saved to {pop_map_path}")

## 6. File Integrity & Hashing

Verify file integrity using cryptographic hashes. These functions are useful for:
- Validating downloaded files against known checksums
- Detecting file corruption or tampering
- Quick change detection for caching/pipeline reprocessing

In [ ]:
from siege_utilities.files.hashing import (
    generate_sha256_hash_for_file,
    get_file_hash,
    verify_file_integrity,
    get_quick_file_signature,
)

print("Hashing functions imported.")

In [ ]:
# Create a sample file and compute hashes
sample_file = OUTPUT_DIR / "hash_demo_sample.txt"
sample_file.write_text(
    "This is a sample file for demonstrating siege_utilities hashing.\n"
    "It contains multiple lines of text to create a realistic test case.\n"
    "File integrity verification is critical for data pipeline reliability.\n"
)
print(f"Created sample file: {sample_file.name} ({sample_file.stat().st_size} bytes)")

# SHA256 hash
sha256_hash = generate_sha256_hash_for_file(sample_file)
print(f"SHA256: {sha256_hash}")

# Multiple algorithms via get_file_hash()
md5_hash = get_file_hash(sample_file, algorithm="md5")
sha1_hash = get_file_hash(sample_file, algorithm="sha1")
sha256_via_generic = get_file_hash(sample_file, algorithm="sha256")

print(f"MD5:    {md5_hash}")
print(f"SHA1:   {sha1_hash}")
print(f"SHA256: {sha256_via_generic}")

assert sha256_hash == sha256_via_generic, "SHA256 hashes should match"
print("Both SHA256 methods produce identical results.")

In [ ]:
# Verify file integrity by comparing against known hashes
print("--- File Integrity Verification ---")

is_valid = verify_file_integrity(sample_file, sha256_hash, algorithm="sha256")
print(f"Correct SHA256:   {is_valid}")

wrong_hash = "0" * 64
is_invalid = verify_file_integrity(sample_file, wrong_hash, algorithm="sha256")
print(f"Incorrect SHA256: {is_invalid}")

is_valid_md5 = verify_file_integrity(sample_file, md5_hash, algorithm="md5")
print(f"Correct MD5:      {is_valid_md5}")

assert is_valid is True
assert is_invalid is False
assert is_valid_md5 is True
print("All integrity checks passed.")

In [ ]:
# Quick file signatures for change detection
print("--- Quick File Signatures ---")

sig_original = get_quick_file_signature(sample_file)
print(f"Original: {sig_original}")

# Modify and show signature changes
sample_file.write_text("This file has been modified to demonstrate change detection.\n")
sig_modified = get_quick_file_signature(sample_file)
print(f"Modified: {sig_modified}")

assert sig_original != sig_modified
print(f"Change detected: {sig_original != sig_modified}")

sig_missing = get_quick_file_signature(OUTPUT_DIR / "does_not_exist.txt")
print(f"Missing file: '{sig_missing}'")

sample_file.unlink(missing_ok=True)
print("Cleaned up demo file.")

## 7. Remote File Downloads

Download files from URLs with progress tracking, retry logic, and metadata inspection.
These utilities handle SSL verification, timeouts, and chunked streaming for large files.

In [ ]:
from siege_utilities.files.remote import (
    download_file,
    download_file_with_retry,
    get_file_info,
    is_downloadable,
)

print("Remote file functions imported.")

In [ ]:
# Check whether URLs are downloadable
TEST_URL = "https://httpbin.org/robots.txt"
INVALID_URL = "https://httpbin.org/status/404"

try:
    can_download = is_downloadable(TEST_URL)
    print(f"is_downloadable(robots.txt): {can_download}")
except Exception as e:
    print(f"Network error: {e}")

try:
    cannot_download = is_downloadable(INVALID_URL)
    print(f"is_downloadable(404 URL):    {cannot_download}")
except Exception as e:
    print(f"Network error: {e}")

In [ ]:
# Inspect remote file metadata without downloading
try:
    info = get_file_info(TEST_URL)
    if info:
        print("Remote file info:")
        for key, value in info.items():
            print(f"  {key}: {value}")
    else:
        print("Could not retrieve file info.")
except Exception as e:
    print(f"Network error: {e}")

In [ ]:
# Download a small public file
download_dest = OUTPUT_DIR / "robots.txt"

try:
    result = download_file(TEST_URL, download_dest, timeout=15)
    if result:
        dest_path = Path(result)
        dl_hash = generate_sha256_hash_for_file(dest_path)
        print(f"Downloaded: {dest_path.name} ({dest_path.stat().st_size} bytes)")
        print(f"SHA256: {dl_hash}")
        dest_path.unlink(missing_ok=True)
    else:
        print("Download returned False.")
except Exception as e:
    print(f"Download skipped: {e}")

In [ ]:
# Download with retry
retry_dest = OUTPUT_DIR / "retry_demo.txt"

try:
    result = download_file_with_retry(
        TEST_URL, retry_dest,
        max_retries=2, retry_delay=1, timeout=15,
    )
    if result:
        dest_path = Path(result)
        print(f"Downloaded with retry: {dest_path.name} ({dest_path.stat().st_size} bytes)")
        dest_path.unlink(missing_ok=True)
    else:
        print("Download with retry returned False.")
except Exception as e:
    print(f"Retry download skipped: {e}")

## Summary

| Section | Function | Purpose |
|---------|----------|---------|
| **Sample Data** | `list_available_datasets()` | List bundled sample datasets |
| | `load_sample_data()` | Load a specific dataset |
| | `generate_synthetic_population()` | Create fake demographic data |
| | `generate_synthetic_businesses()` | Create fake business records |
| | `generate_synthetic_housing()` | Create fake housing data |
| **File Hashing** | `generate_sha256_hash_for_file()` | SHA256 hash with chunked reading |
| | `get_file_hash()` | Hash with configurable algorithm (sha256, md5, sha1, etc.) |
| | `verify_file_integrity()` | Compare file hash against expected value |
| | `get_quick_file_signature()` | Fast change detection (partial hash for large files) |
| **Remote Files** | `is_downloadable()` | Check if a URL serves downloadable content |
| | `get_file_info()` | Inspect remote file metadata (size, type, etag) |
| | `download_file()` | Stream download with progress bar |
| | `download_file_with_retry()` | Download with automatic retry on failure |

**Use Cases:**
- Testing data pipelines without real data
- Developing visualizations with synthetic datasets
- Validating downloaded files against known checksums
- Detecting file corruption or unauthorized changes
- Downloading remote data files with resilient retry logic
- Privacy-safe analytics development